# Dag 5: Sweep Jobs + Pipelines

**Nøglebegreber:** `SweepJob`, `search_space`, `sampling_algorithm`, `early_termination`, `@pipeline`, `PipelineJob`

**Læringsmål:**
- Forstå hvad et Sweep Job er og hvornår det er mere egnet end manuel hyperparameter-søgning
- Definere et `search_space` med diskrete og kontinuerte hyperparametre
- Konfigurere sampling-strategi (`random`, `grid`, `bayesian`) og early termination
- Submitte et Sweep Job og hente det bedste run
- Forstå hvad en Azure ML Pipeline er og hvad den bruges til
- Bygge en to-trins pipeline med `@pipeline`-dekoratoren
- Koble outputs fra et pipeline-trin til inputs i næste trin

**Forudsætninger:** Dag 1 (data assets), Dag 2 (environments + command jobs), Dag 3 (MLflow logging)

## 1. MLClient
Opret forbindelse til workspace

In [1]:
import sys
sys.path.append("..")

from src.utils import init_ml_client
ml_client = init_ml_client()

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## 2. Hvad er et Sweep Job?

Et **Sweep Job** (også kaldet hyperparameter-tuning job) automatiserer søgningen efter de bedste hyperparametre. I stedet for manuelt at submitte ét job per reg-værdi (som i Dag 3 bonus-opgaven), definerer du et **search space** og lader Azure ML prøve kombinationer systematisk.

Et Sweep Job består af:
1. Et **trial command** — det script der køres per forsøg (f.eks. `train.py`)
2. Et **search_space** — de hyperparametre der søges over og deres fordelinger
3. En **sampling_algorithm** — strategi for hvordan kombinationer vælges
4. En **primary_metric** og **goal** — hvad der optimeres og i hvilken retning
5. Valgfri **early_termination** — stop tidligt hvis runs ikke er lovende

I SDK v2 bruges klassen `Sweep` fra `azure.ai.ml.sweep`.

> **Eksamen tip:** Et Sweep Job er et **parent job** med mange **child jobs** (ét per trial). Child-runs logges automatisk som MLflow runs under samme experiment. Det minder om AutoML, men Sweep Jobs giver dig fuld kontrol over modellen og træningskoden — AutoML vælger også algoritmen.

**Opgave:** List alle jobs i workspace og find de sweep-jobs du eventuelt har kørende fra tidligere dage.

*Hint:* `ml_client.jobs.list()` returnerer alle jobs. Filtrer på `type == "sweep"` hvis du vil se kun sweep-jobs.

In [ ]:
# TODO: List de seneste 10 jobs i workspace og print navn, type og status
# HINT: ml_client.jobs.list() er en iterator — brug list() og slice [:10]

for job in ...:
    print(...)

## 3. Definer search space

Search space definerer hvilke hyperparametre der søges over og deres sandsynlighedsfordelinger. I SDK v2 importeres fordelingerne fra `azure.ai.ml.sweep`:

| Fordeling | Beskrivelse | Eksempel |
|-----------|-------------|----------|
| `Choice([a, b, c])` | Diskret — vælg en af de listede værdier | `Choice(["liblinear", "lbfgs"])` |
| `Uniform(min, max)` | Kontinuert uniform fordeling | `Uniform(0.001, 1.0)` |
| `LogUniform(min, max)` | Log-uniform (god til learning rates) | `LogUniform(-4, 0)` |
| `Normal(mu, sigma)` | Normalfordeling | `Normal(0.1, 0.05)` |
| `QUniform(min, max, q)` | Diskretiseret uniform | `QUniform(2, 10, 2)` |
| `Randint(upper)` | Heltal fra 0 til upper | `Randint(100)` |

> **Eksamen tip:** `Choice` bruges til **diskrete** værdier (strings, ints, floats fra en fast liste). `Uniform` og `LogUniform` bruges til **kontinuerte** værdier. `LogUniform(-4, 0)` svarer til `10^(-4)` til `10^0` — god til learning rates og regularisering, da de typisk spænder over mange størrelsesordener.

**Opgave:** Definer et search space for `reg` og `solver` til `train.py`:
- `reg`: kontinuert log-uniform søgning fra `0.001` til `1.0`
- `solver`: diskret valg mellem `"liblinear"` og `"lbfgs"`

*Hint:* `from azure.ai.ml.sweep import Choice, LogUniform` og definer `search_space` som en dict.

In [2]:
from azure.ai.ml.sweep import Choice, LogUniform, Uniform

# TODO: Definer search_space som en dict med nøgler "reg" og "solver"
# HINT: LogUniform tager log10-grænser — LogUniform(-3, 0) svarer til 0.001 til 1.0

search_space = {
    'reg': Choice(values=[0.001, 0.05, 0.1]),
    'solver': Choice(values=['lbfgs', 'liblinear'])
}

print("Search space defineret:", search_space)

Search space defineret: {'reg': <azure.ai.ml.entities._job.sweep.search_space.Choice object at 0x1052418b0>, 'solver': <azure.ai.ml.entities._job.sweep.search_space.Choice object at 0x16a2b5ca0>}


## 4. Tilpas train.py til at acceptere solver-parameter

Det nuværende `src/train.py` accepterer `--reg` men ikke `--solver` som argument. Sweep Job'et sender hyperparametre som kommandolinje-argumenter, så vi skal tilføje `--solver` til `parse_args()`.

> **Eksamen tip:** I et Sweep Job sendes hyperparametre til trial-scriptet via **command-strengens placeholders**, f.eks. `--reg ${{search_space.reg}}`. Azure ML substituerer disse med de aktuelle værdier per trial. Scriptet skal have tilsvarende `argparse`-argumenter.

**Opgave:** Brug `%%writefile` til at opdatere `src/train.py` med:
1. Et nyt argument `--solver` i `parse_args()` med default `"liblinear"`
2. Brug `args.solver` i `LogisticRegression(solver=args.solver)` i `main()`
3. Log `solver` som parameter med `mlflow.log_param("solver", args.solver)`

*Hint:* Kopier det eksisterende script og tilføj de tre ændringer. Pas på ikke at ændre model-signatur eller andre eksisterende features.

In [4]:
%%writefile ../src/train.py
import argparse
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.sklearn
from mlflow.types.schema import Schema, ColSpec
from mlflow.models.signature import ModelSignature

input_schema = Schema([
    ColSpec("integer", "Age"),
    ColSpec("integer", "WorkLifeBalance"),
    ColSpec("integer", "YearsSinceLastPromotion"),
    ColSpec("integer", "JobInvolvement"),
    ColSpec("integer", "YearsAtCompany"),
    ColSpec("integer", "MonthlyIncome"),
    ColSpec("integer", "Gender_Female"),
    ColSpec("integer", "Department_Human Resources"),
    ColSpec("integer", "Department_Research & Development"),
    ColSpec("integer", "Department_Sales"),
])

output_schema = Schema([ColSpec("boolean")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

def make_dummies(df: pd.DataFrame, categorical_columns: list) -> pd.DataFrame:
    for col in categorical_columns:
        dummies = pd.get_dummies(df[col], prefix=col)
        df = pd.concat([df, dummies], axis=1)
    df.drop(columns=categorical_columns, inplace=True)
    return df

def get_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Analyzing {len(df)} rows of data")
    return df

def log_coef_plot(model: LogisticRegression, feature_names: list, output_dir: Path) -> None:
    """Lav et coefficients-plot og log det som MLflow-artefakt."""
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(feature_names, model.coef_[0])
    ax.set_title("Logistic Regression Coefficients")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Coefficient")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plot_path = output_dir / "coef_plot.png"
    plt.savefig(str(plot_path))
    plt.close(fig)
    mlflow.log_artifact(str(plot_path))
    print(f"Coefficient plot logget: {plot_path}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_data", dest="input_data", type=str, required=True)
    parser.add_argument("--reg", dest="reg", type=float, default=0.01)
    parser.add_argument("--model_dir", type=str, required=True)
    parser.add_argument("--solver", type=str, default="liblinear")
    return parser.parse_args()

def main(args: argparse.Namespace) -> None:
    df = get_data(args.input_data)
    keep_cols = [
        "Attrition", "Age", "Gender", "Department", 
        "WorkLifeBalance", "YearsSinceLastPromotion", 
        "JobInvolvement", "YearsAtCompany", "MonthlyIncome"
    ]
    df = df[keep_cols]
    df = make_dummies(df, ["Gender", "Department"])
    X = df.drop(columns=["Attrition"]).values
    y = df["Attrition"].values
    feature_names = df.drop(columns=["Attrition"]).columns.tolist()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
    C = 1.0 / float(args.reg)

    with mlflow.start_run(run_name=f"sweep-lr-reg-{args.reg}"):
        print(f"Training LogisticRegression with reg={args.reg}, C={C}")
        model = LogisticRegression(C=C, solver=args.solver).fit(X_train, y_train)
        y_hat = model.predict(X_test)
        acc = float(np.average(y_hat == y_test))
        print(f"Accuracy: {acc}")
        mlflow.log_param("reg", args.reg)
        mlflow.log_param("solver", args.solver)
        mlflow.log_metric("val_accuracy", acc)

        if len(np.unique(y_test)) == 2:
            y_scores = model.predict_proba(X_test)[:, 1]
            auc = float(roc_auc_score(y_test, y_scores))
            print(f"AUC: {auc}")
            mlflow.log_metric("val_auc", auc)

        out_dir = Path(args.model_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)
        log_coef_plot(model, feature_names, out_dir)

if __name__ == "__main__":
    args = parse_args()
    main(args)

Overwriting ../src/train.py


## 5. Konfigurer og submit Sweep Job

Et Sweep Job i SDK v2 bygges i to trin:

**Trin 1:** Definer et **trial command** — et `command()` job, men med `${{search_space.PARAM}}`-placeholders i command-strengen i stedet for faste værdier.

**Trin 2:** Kald `.sweep()` på trial-kommandoen for at konvertere det til et sweep job.

`.sweep()` tager disse vigtige parametre:

| Parameter | Type | Beskrivelse |
|-----------|------|-------------|
| `sampling_algorithm` | str | `"random"`, `"grid"`, eller `"bayesian"` |
| `primary_metric` | str | Metric-navn der optimeres (skal matche `mlflow.log_metric` nøgle) |
| `goal` | str | `"maximize"` eller `"minimize"` |
| `early_termination` | objekt | F.eks. `BanditPolicy` eller `MedianStoppingPolicy` |

> **Eksamen tip:** Sampling-algoritmer:
> - **`random`** — hurtig, god generel-løsning, understøtter alle fordelinger
> - **`grid`** — udtømmende søgning, kun til `Choice`-fordelinger, kan blive dyrt
> - **`bayesian`** — intelligent søgning baseret på tidligere runs, kræver `>= 20` trials for at virke godt, understøtter **ikke** early termination
>
> `primary_metric`-strengen skal matche **præcis** den nøgle du bruger i `mlflow.log_metric()` i træningsscriptet.

**Opgave:** Byg og submit et Sweep Job:
1. Definer et trial `command()` job med `${{search_space.reg}}` og `${{search_space.solver}}` i command-strengen
2. Kald `.sweep()` med `sampling_algorithm="random"`, `primary_metric="val_auc"`, `goal="maximize"`
3. Sæt `max_total_trials=8` og `max_concurrent_trials=2` på sweep-jobbet
4. Submit med `ml_client.jobs.create_or_update()`

*Hint:* Trial command laves med `command()` som normalt, men command-strengen bruger `${{search_space.reg}}` i stedet for en fast værdi.

In [ ]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

# Trin 1: Definer trial command
# TODO: Definer et command() job der kører train.py med search_space placeholders
# HINT: command-strengen skal indeholde --reg ${{search_space.reg}} og --solver ${{search_space.solver}}

trial_command = ...  # din kode her

# Trin 2: Konverter til sweep job
# TODO: Kald .sweep() på trial_command med sampling_algorithm, primary_metric og goal
# HINT: sweep_job = trial_command.sweep(sampling_algorithm=..., primary_metric=..., goal=...)

sweep_job = ...  # din kode her

# Trin 3: Sæt limits
# TODO: Sæt sweep_job.set_limits(max_total_trials=8, max_concurrent_trials=2)

# Trin 4: Submit
# TODO: Submit sweep_job og print job-navn og Studio URL

returned_sweep = ...  # din kode her
print(f"Sweep job: {returned_sweep.name}")
print(f"Studio URL: {returned_sweep.studio_url}")

## 6. Early Termination

Early termination stopper trials der sandsynligvis ikke vil forbedre resultatet, og sparer dermed compute-tid og penge. Azure ML understøtter flere politikker:

| Politik | Beskrivelse |
|---------|-------------|
| `BanditPolicy` | Stop trials der er mere end X% dårligere end det bedste run |
| `MedianStoppingPolicy` | Stop trials der er dårligere end medianen af alle runs |
| `TruncationSelectionPolicy` | Stop de bundne X% af trials |

`BanditPolicy` er den mest brugte og forstår to parametre:
- `slack_factor` — f.eks. `0.1` betyder et trial stoppes hvis det er mere end 10% dårligere end bedste run
- `evaluation_interval` — evaluer hvert N-te step
- `delay_evaluation` — vent X steps før første evaluering (giv runs tid til at varme op)

> **Eksamen tip:** Early termination kræver at scriptet logger metrics med `step`-parameteren (eller at metrics logges kontinuert, f.eks. per epoch). For en enkelt-metric log uden steps er `MedianStoppingPolicy` mere passende. `BanditPolicy` og `MedianStoppingPolicy` er **ikke** kompatible med `bayesian` sampling — brug dem kun med `random` eller `grid`.

**Opgave:** Opret et nyt sweep job (kopi af forrige) men tilføj en `BanditPolicy` med `slack_factor=0.1` og `evaluation_interval=1`.

*Hint:* `from azure.ai.ml.sweep import BanditPolicy` — angiv den som `early_termination=BanditPolicy(...)` i `.sweep()`-kaldet.

In [ ]:
from azure.ai.ml.sweep import BanditPolicy

# TODO: Opret en BanditPolicy med slack_factor=0.1 og evaluation_interval=1
early_termination_policy = ...  # din kode her

# TODO: Genbyg sweep_job men tilføj early_termination=early_termination_policy i .sweep()-kaldet
# HINT: Du kan genbruge trial_command fra forrige celle — bare kald .sweep() igen med den nye parameter

sweep_job_with_et = ...  # din kode her
sweep_job_with_et.set_limits(max_total_trials=8, max_concurrent_trials=2)

# Udkommenter linjen nedenfor når du er klar til at submitte
# returned_sweep_et = ml_client.jobs.create_or_update(sweep_job_with_et)
# print(f"Sweep job (med early termination): {returned_sweep_et.name}")
# print(f"Studio URL: {returned_sweep_et.studio_url}")

## 7. Hent det bedste trial fra Sweep Job

Når Sweep Job'et er fuldført, kan du programmatisk hente det bedste child-run via MLflow. Alle trials er MLflow-runs under samme experiment.

Du kan også hente best trial direkte via job-objektet:
- `ml_client.jobs.get(returned_sweep.name)` returnerer det opdaterede job-objekt
- Det bedste trial kan tilgås som et child job under parent-jobbet

> **Eksamen tip:** I Azure ML Studio kan du se alle trials under Sweep Job'et i fanebladet "Trials". Her kan du sammenligne metrics visuelt og se hvilke hyperparametre der korrelerer med den bedste performance. `mlflow.search_runs()` med `filter_string` og `order_by` giver den samme information programmatisk.

**Opgave:** Vent på at sweep job'et er færdigt og hent derefter det bedste run:
1. Stream logs med `ml_client.jobs.stream()`
2. Hent alle runs fra experiment `"attrition-sweep"` sorteret efter `val_auc` (faldende)
3. Print `run_id`, `params.reg`, `params.solver` og `metrics.val_auc` for de 5 bedste

*Hint:* `mlflow.search_runs(experiment_names=["attrition-sweep"], order_by=["metrics.val_auc DESC"])`

In [ ]:
import mlflow

# Sæt MLflow tracking URI
tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)

# TODO: Stream logs fra det indsendte sweep job
# ADVARSEL: Sweep jobs kan tage 10-20 minutter
# HINT: ml_client.jobs.stream(returned_sweep.name)


In [ ]:
import pandas as pd

# TODO: Søg alle runs i "attrition-sweep" eksperimentet og vis de 5 bedste på val_auc
# HINT: mlflow.search_runs(experiment_names=[...], order_by=[...], max_results=...)

sweep_runs = ...  # din kode her

# TODO: Vis run_id, params.reg, params.solver og metrics.val_auc for de 5 bedste
cols = ["run_id", "params.reg", "params.solver", "metrics.val_auc"]
print(sweep_runs[cols].head(5).to_string(index=False))

## 8. Hvad er en Azure ML Pipeline?

En **Pipeline** er en sekvens af trin (steps) der udføres i en defineret rækkefølge. Hvert trin er typisk et command job — men samlet udgør de en **reproducerbar, orkestreret workflow**.

Pipelines bruges til:
- **Reproducerbarhed** — samme pipeline kan køres igen med nye data og give samme resultat
- **Modularitet** — hvert trin kan opdateres uafhængigt
- **Parallelisme** — trin der ikke afhænger af hinanden kan køre parallelt
- **Datadeling** — outputs fra ét trin kobles automatisk som inputs til næste

I SDK v2 defineres pipelines med `@pipeline`-dekoratoren fra `azure.ai.ml.dsl`.

**Grundlæggende struktur:**
```python
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import load_component

@pipeline
def my_pipeline(pipeline_input_data):
    step1 = component_1(data=pipeline_input_data)
    step2 = component_2(data=step1.outputs.processed_data)
    return {"final_model": step2.outputs.model}
```

> **Eksamen tip:** I SDK v2 defineres pipeline-trin typisk som **components** (`CommandComponent`) eller ved at kalde `command()` direkte inde i pipeline-funktionen. Pipeline-funktionen modtager inputs og returnerer outputs. Azure ML afvikler automatisk trin i korrekt rækkefølge baseret på data-afhængigheder.

**Opgave:** Svar på disse spørgsmål (i en ny markdown-celle nedenfor) inden du fortsætter:
1. Hvad er forskellen på et Pipeline-trin og et standalone Command Job?
2. Hvornår giver det mening at bruge en pipeline frem for at submitte jobs separat?

*(Skriv dine svar her)*

## 9. Byg en to-trins pipeline

Vi bygger en simpel pipeline med to trin:
- **Trin 1 (`prep`)**: Henter rådata og gemmer en forbehandlet version (subset af kolonner, dummy-kodning)
- **Trin 2 (`train`)**: Modtager det forbehandlede data fra trin 1 og træner en model

Hvert trin defineres som et **command component** med `CommandComponent`. Pipeline-funktionen kobler dem sammen.

Først opretter vi trin 1: et prep-script der forbehandler data.

> **Eksamen tip:** Output fra ét pipeline-trin refereres i næste trin som `step1.outputs.output_name`. Azure ML opretter automatisk en midlertidig datastore-sti til disse mellemliggende data. Du behøver ikke angive stien manuelt — Azure ML håndterer dataflowet.

**Opgave:** Opret et preprocessering-script `src/prep.py` der:
1. Accepterer `--input_data` (sti til rå CSV) og `--output_data` (sti til outputmappe)
2. Indlæser CSV-filen, beholder relevante kolonner og laver dummy-encoding
3. Gemmer det forbehandlede datasæt som `prep_output.csv` i `--output_data` mappen
4. Logger antal rækker og kolonner med `mlflow`

*Hint:* Brug `make_dummies`-logikken fra `train.py`. Output-mappen oprettes med `Path(args.output_data).mkdir(parents=True, exist_ok=True)`.

In [ ]:
%%writefile ../src/prep.py
import argparse
from pathlib import Path
import pandas as pd
import mlflow


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    # TODO: Tilføj --input_data og --output_data argumenter
    return parser.parse_args()


def main(args: argparse.Namespace) -> None:
    # TODO: Indlæs CSV fra args.input_data
    # TODO: Behold disse kolonner: Attrition, Age, Gender, Department,
    #        WorkLifeBalance, YearsSinceLastPromotion, JobInvolvement,
    #        YearsAtCompany, MonthlyIncome
    # TODO: Lav dummy-encoding af Gender og Department
    # TODO: Log antal rækker og kolonner med mlflow.log_metric()
    # TODO: Gem forbehandlet data til Path(args.output_data) / "prep_output.csv"
    # HINT: Path(args.output_data).mkdir(parents=True, exist_ok=True)
    pass


if __name__ == "__main__":
    args = parse_args()
    main(args)

## 10. Definer pipeline med @pipeline-dekoratoren

Nu definerer vi selve pipeline-funktionen der koordinerer de to trin. I SDK v2 bruges `@pipeline`-dekoratoren til at markere en funktion som en pipeline-definition.

Inde i pipeline-funktionen definerer du hvert trin som en `command()`-konfiguration. Output fra trin 1 sendes som input til trin 2 via `trin1.outputs.output_navn`.

> **Eksamen tip:** Når du definerer trin inde i en `@pipeline`-funktion, bruges `command()` på næsten samme måde som til standalone jobs — men `compute` kan sættes globalt på pipeline-niveau og arves af alle trin med `default_compute`. Input/output-kobling sker ved at referere til `trin.outputs.navn`.

**Opgave:** Definer en pipeline-funktion `attrition_pipeline(raw_data)` der:
1. Kører trin 1 (`prep_step`): `src/prep.py` med `raw_data` som input
2. Kører trin 2 (`train_step`): `src/train.py` med output fra `prep_step` som input og `reg=0.05`, `solver="liblinear"`
3. Returnerer `{"trained_model": train_step.outputs.model_dir}`

Derefter: Instansier og submit pipeline-jobbet.

*Hint:* `from azure.ai.ml.dsl import pipeline` — dekorér din funktion med `@pipeline(default_compute="my-cluster")`.

In [ ]:
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

# TODO: Definer pipeline-funktionen med @pipeline-dekoratoren
# HINT: @pipeline(default_compute="my-cluster", experiment_name="attrition-pipeline")

# TODO: Inde i funktionen: definer prep_step og train_step som command() kald
# HINT: train_step får input fra prep_step.outputs.output_data

# TODO: Instansier pipeline-jobbet ved at kalde funktionen med et Input-objekt
# HINT: pipeline_job = attrition_pipeline(raw_data=Input(type=AssetTypes.URI_FILE, path="azureml:ibm-churn-file:1"))

# TODO: Submit pipeline-jobbet med ml_client.jobs.create_or_update()
# TODO: Print Studio URL

pipeline_job = ...  # instansiér pipeline her

returned_pipeline = ...  # submit her
print(f"Pipeline job: {returned_pipeline.name}")
print(f"Studio URL: {returned_pipeline.studio_url}")

In [ ]:
# TODO: Stream pipeline-logs og vent på completion
# HINT: ml_client.jobs.stream(returned_pipeline.name)


## 11. Inspicér pipeline i Azure ML Studio

Når pipeline-jobbet kører (eller er fuldført), kan du åbne det i Azure ML Studio og se:
- Et grafisk view af de to trin og dataflydet imellem dem
- Individuelle logs for hvert trin
- Outputs fra hvert trin i den mellemliggende datastore

> **Eksamen tip:** I Azure ML Studio vises pipeline-jobs under **Jobs** med et særligt pipeline-ikon. Klik på en node i grafvisningen for at se logs, metrics og outputs for det specifikke trin. Outputs fra trin gemmes automatisk i `workspaceblobstore` under en genereret sti.

**Opgave:** Åbn pipeline-jobbet i Azure ML Studio via `studio_url`. Find:
1. Hvilke compute-ressourcer kørte hvert trin på?
2. Hvad er output-stien fra prep-trinnet (den sti der sendes videre til train-trinnet)?
3. Hvilke MLflow-metrics loggede hvert trin?

Skriv dine observationer i markdown-cellen nedenfor.

*(Skriv dine observationer her)*

## 12. (Bonus) Tilføj Sweep Job som pipeline-trin

En avanceret brug af pipelines er at kombinere preprocessering med hyperparameter-tuning i én pipeline. I stedet for et fast `train_step` kan du definere et sweep-trin der automatisk finder de bedste hyperparametre.

Dette gøres ved at kalde `.sweep()` på et command-trin **inde i** pipeline-funktionen.

> **Eksamen tip:** Sweep Jobs kan inlejres i pipelines ved at kalde `.sweep(sampling_algorithm=..., primary_metric=..., goal=...)` på et command-trin inde i `@pipeline`-funktionen. Det inlejrede sweep-trin opfører sig som et normalt pipeline-trin — det modtager inputs og producerer outputs — men afvikler internt mange trials.

**Opgave:** Byg en ny pipeline `attrition_pipeline_with_sweep(raw_data)` der:
1. Har det samme `prep_step` som før
2. Erstatter `train_step` med et sweep-trin der søger over `reg` (LogUniform) og `solver` (Choice)
3. Sætter `max_total_trials=6` og `primary_metric="val_auc"` med `goal="maximize"`

*Hint:* Definer train-kommandoen inde i pipeline-funktionen, og kald derefter `.sweep(...)` på den, `.set_limits(max_total_trials=6)` på sweep-objektet.

In [ ]:
from azure.ai.ml.sweep import Choice, LogUniform, BanditPolicy
from azure.ai.ml.dsl import pipeline

# TODO: Definer en pipeline der kombinerer prep_step + sweep-baseret train_step
# HINT: Inde i @pipeline-funktionen:
#   1. Definer prep_step som før
#   2. Definer train_command = command(...) med search_space placeholders i command-strengen
#   3. sweep_step = train_command.sweep(sampling_algorithm="random", primary_metric="val_auc", goal="maximize")
#   4. sweep_step.set_limits(max_total_trials=6)

# Udkommenter og submit når du er klar
# pipeline_sweep_job = attrition_pipeline_with_sweep(
#     raw_data=Input(type=AssetTypes.URI_FILE, path="azureml:ibm-churn-file:1")
# )
# returned = ml_client.jobs.create_or_update(pipeline_sweep_job)
# print(f"Pipeline + Sweep job: {returned.name}")
# print(f"Studio URL: {returned.studio_url}")

## Refleksion

Besvar disse spørgsmål med dine egne ord:

1. Hvad er forskellen på et Sweep Job og det manuelle loop med tre command jobs fra Dag 3 bonus-opgaven? Hvad vinder du ved at bruge Sweep?

2. Hvornår ville du vælge `bayesian` sampling frem for `random` i et Sweep Job?

3. Hvorfor er early termination ikke kompatibel med `bayesian` sampling?

4. Hvad er fordelen ved at definere preprocessering som et separat pipeline-trin frem for at inkludere det i træningsscriptet?

5. Hvad sker der med mellemliggende data (f.eks. `prep_output.csv`) i en pipeline — hvor gemmes det, og hvem styrer stien?

*(Skriv dine svar her)*

## Nøglepunkter (DP-100 eksamen)

- **Sweep Job vs. manuel loop**: Sweep Jobs automatiserer hyperparameter-søgning med built-in sampling, parallel execution og early termination. Et Sweep Job er et parent job med mange child-runs logget som MLflow runs.
- **`search_space`**: Dict med hyperparameter-fordelinger. `Choice([...])` til diskrete værdier, `Uniform(min, max)` og `LogUniform(min, max)` til kontinuerte. `LogUniform` foretrækkes til learning rates og regularisering.
- **Sampling-algoritmer**: `random` (generel, understøtter alle fordelinger), `grid` (udtømmende, kun `Choice`), `bayesian` (intelligent, kræver mange trials, ingen early termination).
- **`primary_metric`**: Skal matche præcis den nøgle der bruges i `mlflow.log_metric()` i træningsscriptet. `goal` er `"maximize"` eller `"minimize"`.
- **Early termination**: `BanditPolicy(slack_factor=X)` stopper trials der er mere end X% dårligere end bedste run. Kun kompatibel med `random` og `grid` sampling. `MedianStoppingPolicy` er alternativet.
- **`@pipeline`-dekoratoren**: Definerer en pipeline-funktion i `azure.ai.ml.dsl`. Trin defineres som `command()`-kald inde i funktionen. Outputs kobles ved `trin.outputs.navn`.
- **Pipeline outputs**: Mellemliggende data gemmes automatisk i `workspaceblobstore`. Du behøver ikke angive stier — Azure ML håndterer dataflowet baseret på output/input-koblingen.
- **Sweep i pipeline**: Et command-trin inde i en pipeline kan konverteres til et sweep-trin med `.sweep()`. Det muliggør hyperparameter-tuning som et orkestreret pipeline-trin.
- **Fase 3 eksamenssektionen** (Train and Deploy Models, 25-30%) — Sweep Jobs og Pipelines er centrale komponenter her og kombineres typisk med model-registrering (Dag 7).